# ZeroMRG Gate 1 COV-CTR 0%-paired preflight

Kaggle-only execution. No held-out test generation or paired experiments. The epoch-90 latest artifact is valid, but the saved best file is epoch 8, whereas history selects epoch 85. This notebook reports/refuses that mismatch. No epoch-85 weights are claimed verified.

Set the exact checkpoint, artifact, best-checkpoint, and raw-dataset paths in the configuration cell. Dependencies and vocabulary are reused, never rebuilt. The embedded runtime is generated from scripts/zeromrg_gate1_runtime.py. Legacy checkpoints lack RNG state and exact encoder revisions; bit-for-bit continuation is unavailable.

In [ ]:
"""Checkpoint-producing Kaggle decoder and fail-closed recovery utilities.

This profile preserves the recorded COV-CTR experiment, not modular defaults.
No training, model download, or inference is performed on import.
"""
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import random
import re
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

ARCHITECTURE = dict(vocab_size=368, max_tokens=160, d_model=768,
                    num_layers=6, num_heads=8, ffn_dim=3072, dropout=0.1,
                    activation="gelu", norm_first=False, final_layer_norm=False,
                    scale_token_embeddings=False, pad_id=0, bos_id=1, eos_id=2)
TRAINING = dict(seed=42, batch_size=6, learning_rate=0.001, momentum=0.99,
                gradient_clip=5.0, patience=50, max_epochs=200,
                precision="float32", scheduler=None,
                selection_metric="validation_text_reconstruction_BLEU4_PTB_x100")
MCLIP_NAME = "M-CLIP/XLM-Roberta-Large-Vit-L-14"
DEPENDENCIES = ("train_manifest.csv", "validation_manifest.csv", "test_manifest.csv",
                "prompt_manifest.csv", "validated_manifest.csv", "validation_audit.json",
                "decoder_vocabulary.json", "mclip_text_features.pt", "mclip_prompt_bank.pt")


def require(condition, message):
    if not condition:
        raise ValueError(message)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_checkpoint(path):
    require(Path(path).is_file(), f"Selected checkpoint does not exist: {path}")
    # No unsafe unpickling fallback. Legacy project artifacts load with weights_only=True.
    return torch.load(path, map_location="cpu", weights_only=True)


class CheckpointReportDecoder(nn.Module):
    def __init__(self, architecture=None):
        super().__init__()
        self.architecture = dict(ARCHITECTURE if architecture is None else architecture)
        a = self.architecture
        require(not a["scale_token_embeddings"] and not a["final_layer_norm"],
                "This class implements the recorded Kaggle decoder only")
        self.token_embedding = nn.Embedding(a["vocab_size"], a["d_model"], padding_idx=a["pad_id"])
        self.position_embedding = nn.Embedding(a["max_tokens"], a["d_model"])
        layer = nn.TransformerDecoderLayer(
            d_model=a["d_model"], nhead=a["num_heads"], dim_feedforward=a["ffn_dim"],
            dropout=a["dropout"], activation=a["activation"], batch_first=True,
            norm_first=a["norm_first"])
        self.transformer = nn.TransformerDecoder(layer, num_layers=a["num_layers"])
        self.output_projection = nn.Linear(a["d_model"], a["vocab_size"])

    def forward(self, condition, input_ids):
        a = self.architecture
        require(input_ids.ndim == 2 and 0 < input_ids.shape[1] <= a["max_tokens"],
                "Decoder IDs must be [B,T] within the saved positional limit")
        require(tuple(condition.shape) == (input_ids.shape[0], a["d_model"]),
                "Condition dimensions incompatible with checkpoint decoder")
        positions = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)
        target = self.token_embedding(input_ids) + self.position_embedding(positions)
        mask = torch.triu(torch.ones(input_ids.shape[1], input_ids.shape[1],
                                    dtype=torch.bool, device=input_ids.device), diagonal=1)
        hidden = self.transformer(target, condition.unsqueeze(1).to(target.dtype),
                                  tgt_mask=mask, tgt_key_padding_mask=input_ids.eq(a["pad_id"]))
        return self.output_projection(hidden)

    @torch.no_grad()
    def generate(self, condition, max_length=None):
        self.eval()
        a = self.architecture
        limit = a["max_tokens"] if max_length is None else int(max_length)
        require(2 <= limit <= a["max_tokens"], "Invalid generation length")
        ids = torch.full((len(condition), 1), a["bos_id"], dtype=torch.long, device=condition.device)
        finished = torch.zeros(len(condition), dtype=torch.bool, device=condition.device)
        for _ in range(limit - 1):
            next_id = self(condition, ids)[:, -1].argmax(-1)
            next_id = torch.where(finished, torch.full_like(next_id, a["pad_id"]), next_id)
            ids = torch.cat((ids, next_id[:, None]), dim=1)
            finished |= next_id.eq(a["eos_id"])
            if bool(finished.all()):
                break
        return ids


def validate_model_state(checkpoint, model):
    require(isinstance(checkpoint, dict), "Checkpoint must be a state dictionary payload")
    state = checkpoint.get("decoder")
    require(isinstance(state, dict), "Required decoder state dictionary absent")
    expected = model.state_dict()
    require(set(state) == set(expected),
            f"Decoder key mismatch: missing={sorted(set(expected)-set(state))}; unexpected={sorted(set(state)-set(expected))}")
    for key, tensor in state.items():
        require(isinstance(tensor, torch.Tensor) and tensor.shape == expected[key].shape,
                f"Decoder shape mismatch: {key}")
        require(tensor.dtype == expected[key].dtype, f"Decoder dtype mismatch: {key}")
        require(bool(torch.isfinite(tensor).all()), f"Non-finite decoder weights: {key}")
    if "architecture" in checkpoint:
        require(checkpoint["architecture"] == model.architecture, "Saved architecture metadata differs")
    for key, expected_value in (("vocabulary_size", model.architecture["vocab_size"]),
                                ("max_tokens", model.architecture["max_tokens"])):
        if key in checkpoint:
            require(checkpoint[key] == expected_value, f"Saved {key} differs")


def inspect_training_state(checkpoint, model, best_checkpoint=None):
    require(isinstance(checkpoint, dict), "Training checkpoint must be a dictionary payload")
    required = {"epoch", "decoder", "optimizer", "history", "best_validation_bleu4", "epochs_without_improvement"}
    require(required <= checkpoint.keys(), f"Required training state absent: {sorted(required-checkpoint.keys())}")
    epoch = checkpoint["epoch"]
    require(type(epoch) is int and epoch > 0, "Saved epoch must be a positive integer")
    require(checkpoint.get("scheduler") is None or checkpoint.get("scheduler") == {},
            "Unknown scheduler state; supply its verified implementation before resuming")
    if "training_config" in checkpoint:
        require(checkpoint["training_config"] == TRAINING, "Training configuration differs from recorded run")
    history = checkpoint["history"]
    require(isinstance(history, list) and len(history) == epoch, "History length differs from saved epoch")
    require(all(isinstance(row, dict) and {"epoch", "training_loss", "validation_bleu4"} <= row.keys()
                for row in history), "History record fields absent or invalid")
    require([row["epoch"] for row in history] == list(range(1, epoch + 1)), "History is not contiguous")
    for row in history:
        require(all(math.isfinite(float(row[key])) for key in ("training_loss", "validation_bleu4")),
                "Non-finite history value")
    selected = max(history, key=lambda row: row["validation_bleu4"])
    require(math.isclose(float(selected["validation_bleu4"]), float(checkpoint["best_validation_bleu4"]),
                         rel_tol=0, abs_tol=1e-10), "History and saved best metric disagree")
    require(type(checkpoint["epochs_without_improvement"]) is int and
            checkpoint["epochs_without_improvement"] == epoch - selected["epoch"], "Patience state disagrees with history")
    validate_model_state(checkpoint, model)
    if best_checkpoint is not None:
        validate_selected_best(checkpoint, model, best_checkpoint)
    opt = checkpoint["optimizer"]
    require(isinstance(opt, dict) and {"state", "param_groups"} <= opt.keys(), "Invalid optimizer state")
    require(len(opt["param_groups"]) == 1, "Expected one decoder SGD parameter group")
    group = opt["param_groups"][0]
    for key, expected in dict(lr=0.001, momentum=0.99, dampening=0, weight_decay=0,
                              nesterov=False, maximize=False).items():
        require(group.get(key, False if key == "maximize" else None) == expected,
                f"SGD setting mismatch: {key}")
    parameters = list(model.named_parameters())
    saved_ids = group["params"]
    require(len(saved_ids) == len(parameters) and len(set(saved_ids)) == len(saved_ids),
            "Optimizer parameter count or mapping incompatible")
    require(set(opt["state"]) == set(saved_ids), "Missing or unexpected optimizer parameter states")
    nonzero = 0
    for pid, (name, parameter) in zip(saved_ids, parameters):
        require(set(opt["state"][pid]) == {"momentum_buffer"}, f"Unknown/missing SGD state: {name}")
        buffer = opt["state"][pid]["momentum_buffer"]
        require(isinstance(buffer, torch.Tensor) and buffer.shape == parameter.shape and
                buffer.dtype == parameter.dtype, f"Momentum shape/dtype incompatible: {name}")
        require(bool(torch.isfinite(buffer).all()), f"Non-finite momentum: {name}")
        nonzero += int(bool(buffer.ne(0).any()))
    require(nonzero > 0, "All SGD momentum is zero")
    if "parameter_names" in checkpoint:
        require(checkpoint["parameter_names"] == [name for name, _ in parameters], "Optimizer parameter order differs")
    return dict(epoch=epoch, next_epoch=epoch + 1, best_epoch=selected["epoch"],
                best_validation_bleu4=float(selected["validation_bleu4"]),
                epochs_without_improvement=checkpoint["epochs_without_improvement"],
                momentum_buffers=len(saved_ids), nonzero_momentum_buffers=nonzero,
                parameter_count=sum(p.numel() for _, p in parameters),
                legacy_rng_missing="rng_state" not in checkpoint,
                scheduler="none", compatibility="PASS")


def validate_selected_best(checkpoint, model, best_checkpoint):
    require(isinstance(best_checkpoint, dict), "Selected best must be a dictionary payload")
    selected = max(checkpoint["history"], key=lambda row: row["validation_bleu4"])
    require(best_checkpoint.get("epoch") == selected["epoch"] and
            math.isclose(float(best_checkpoint.get("validation_bleu4", float("nan"))),
                         float(selected["validation_bleu4"]), rel_tol=0, abs_tol=1e-10),
            f"Best checkpoint disagrees with history: need epoch {selected['epoch']}, "
            f"found epoch {best_checkpoint.get('epoch')}. Recover matching weights; do not substitute latest weights.")
    validate_model_state(best_checkpoint, model)


def restore_weights_optimizer(checkpoint, model, device="cpu"):
    """Restore latest state for diagnostics; not authorization to continue training."""
    result = inspect_training_state(checkpoint, model)
    model.load_state_dict(checkpoint["decoder"], strict=True)
    model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.99)
    optimizer.load_state_dict(checkpoint["optimizer"])
    for parameter, state in optimizer.state.items():
        state["momentum_buffer"] = state["momentum_buffer"].to(parameter.device)
    for pid, parameter in zip(checkpoint["optimizer"]["param_groups"][0]["params"], model.parameters()):
        require(torch.equal(optimizer.state[parameter]["momentum_buffer"].cpu(),
                            checkpoint["optimizer"]["state"][pid]["momentum_buffer"].cpu()),
                "Momentum changed during restoration")
    model.eval()
    return optimizer, result


def restore_training(checkpoint, model, best_checkpoint, device="cpu"):
    result = inspect_training_state(checkpoint, model, best_checkpoint)
    optimizer, _ = restore_weights_optimizer(checkpoint, model, device)
    return optimizer, result


def read_manifest(directory, name):
    with (Path(directory) / name).open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def validate_artifacts(directory, expected_hashes=None):
    directory = Path(directory)
    hashes = {name: sha256(directory / name) for name in DEPENDENCIES}
    if expected_hashes is not None:
        require(hashes == expected_hashes, "Feature/vocabulary/split dependency hashes differ")
    records = {name: read_manifest(directory, name + "_manifest.csv")
               for name in ("train", "validation", "test", "prompt", "validated")}
    ids = {}
    for name, count in dict(train=571, validation=71, test=72, prompt=500, validated=714).items():
        rows = records[name]
        ids[name] = [row["image_id"] for row in rows]
        require(len(rows) == count and len(set(ids[name])) == count, f"{name} count/ID uniqueness failed")
    train, val, test = (set(ids[name]) for name in ("train", "validation", "test"))
    require(not (train & val or train & test or val & test), "Split leakage")
    require(train | val | test == set(ids["validated"]), "Split population differs")
    require(set(ids["prompt"]) <= train, "Prompt bank contains non-training IDs")
    by_id = {r["image_id"]: r for r in records["validated"]}
    for name in ("train", "validation", "test", "prompt"):
        require(all(row["report"] == by_id[row["image_id"]]["report"] for row in records[name]),
                f"{name} reports differ from validated manifest")
    audit = json.loads((directory / "validation_audit.json").read_text(encoding="utf-8"))
    require(audit["csv_rows"] == 726 and audit["physical_images"] == 746, "Raw audit population differs")
    vocabulary = json.loads((directory / "decoder_vocabulary.json").read_text(encoding="utf-8"))
    require(vocabulary["max_tokens"] == 160 and len(vocabulary["vocabulary"]) == 368,
            "Vocabulary or positional capacity differs")
    require(vocabulary["special_token_ids"] == dict(pad=0, bos=1, eos=2, unk=3), "Special-token IDs differ")
    expected_vocab = ["<pad>", "<bos>", "<eos>", "<unk>"] + sorted({
        token for row in records["train"] for token in re.findall(r"\w+|[^\w\s]", row["report"])})
    require(vocabulary["vocabulary"] == expected_vocab, "Vocabulary is not the saved training-only lexicographic mapping")
    text = load_checkpoint(directory / "mclip_text_features.pt")
    require(text["model"] == MCLIP_NAME and text["normalization"] == "l2", "Text encoder metadata differs")
    for name in ("train", "validation"):
        require(set(text[name]) == set(ids[name]), f"{name} feature membership differs")
        for vector in text[name].values():
            require(vector.shape == (768,) and bool(torch.isfinite(vector).all()) and
                    torch.allclose(vector.norm(), torch.tensor(1.0), atol=1e-4), "Invalid normalized text feature")
    prompt = load_checkpoint(directory / "mclip_prompt_bank.pt")
    require(prompt["model"] == MCLIP_NAME and prompt.get("normalization") == "raw", "Prompt encoder/normalization differs")
    require(prompt["prompt_ids"] == ids["prompt"], "Prompt tensor row order/provenance differs")
    vectors = prompt["embeddings"]
    require(vectors.shape == (500, 768) and bool(torch.isfinite(vectors).all()) and
            bool((vectors.norm(dim=-1) > 0).all()), "Invalid prompt bank")
    aligned = torch.stack([text["train"][key] for key in ids["prompt"]])
    # Separate encodings are not bitwise equal. Record the observed difference;
    # membership and frozen-cache hashes above are the provenance assertions.
    normalized_prompts = F.normalize(vectors, dim=-1)
    cosine = (normalized_prompts * aligned).sum(-1)
    cache_comparison = dict(max_absolute_difference=float((normalized_prompts - aligned).abs().max()),
                            minimum_cosine=float(cosine.min()), mean_cosine=float(cosine.mean()),
                            exact_encoder_revision="UNVERIFIED")
    return dict(records=records, vocabulary=vocabulary, text=text, prompt=prompt,
                dependency_hashes=hashes, prompt_text_comparison=cache_comparison)


def encode_report(text, vocabulary):
    lookup = {token: i for i, token in enumerate(vocabulary)}
    ids = [1] + [lookup.get(token, 3) for token in re.findall(r"\w+|[^\w\s]", text)] + [2]
    require(len(ids) <= 160, "Report exceeds saved positional capacity; no silent truncation")
    return ids


def decode_report(token_ids, vocabulary):
    tokens = []
    for raw in token_ids:
        token = int(raw)
        if token == 2:
            break
        require(0 <= token < len(vocabulary), "Generated token outside vocabulary")
        if token not in {0, 1}:
            tokens.append(vocabulary[token])
    text = re.sub(r"\s+([.,!?;:%)\]])", r"\1", " ".join(tokens))
    return re.sub(r"([(\[])\s+", r"\1", text).strip()


def enrich(image, prompts):
    require(image.ndim == prompts.ndim == 2 and image.shape[-1] == prompts.shape[-1] == 768,
            "Image/prompt dimensions must share the recorded 768-D CLIP space")
    require(bool(torch.isfinite(image).all()) and bool(torch.isfinite(prompts).all()), "Non-finite encoder output")
    scores = image @ prompts.T / math.sqrt(768)
    attention = scores.softmax(-1)
    residual = image + attention @ prompts
    require(bool((residual.norm(dim=-1) > 0).all()), "Zero residual embedding")
    phi = F.normalize(residual, p=2, dim=-1)
    require(bool(torch.isfinite(phi).all()), "Non-finite enriched features")
    return phi, attention


def capture_rng():
    return dict(python=random.getstate(), torch_cpu=torch.get_rng_state(),
                torch_cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])


def restore_rng(state):
    random.setstate(state["python"])
    torch.set_rng_state(state["torch_cpu"].cpu())
    if state["torch_cuda"]:
        require(len(state["torch_cuda"]) == torch.cuda.device_count(), "CUDA RNG device count differs")
        torch.cuda.set_rng_state_all([value.cpu() for value in state["torch_cuda"]])


def atomic_save(payload, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    with temporary.open("wb") as handle:
        torch.save(payload, handle)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, destination)


def training_payload(model, optimizer, epoch, history, best_metric, bad_epochs, hashes, provenance):
    return dict(schema_version="zeromrg_covctr_gate1_v1", epoch=epoch, decoder=model.state_dict(),
                optimizer=optimizer.state_dict(), scheduler=None, history=history,
                best_validation_bleu4=best_metric, epochs_without_improvement=bad_epochs,
                architecture=model.architecture, parameter_names=[n for n, _ in model.named_parameters()],
                training_config=TRAINING, dependency_hashes=hashes, provenance=provenance,
                rng_state=capture_rng())

In [ ]:
CANONICAL_DEPENDENCY_HASHES = {'train_manifest.csv': '8902b8fea1163429aec6940a9744f7c792d1b7f9a00ca300a1fab17448ead973', 'validation_manifest.csv': '94c555cb25a19485616065079139722bfe7122a17639267ba5e03d9e4cfa7c5d', 'test_manifest.csv': 'fcbcd1bd12220b0a3a286c4b1a82a8543e34c0a3495c5d0b8c89c81033b3eb66', 'prompt_manifest.csv': 'f60c45a1dc9beecba16917d767f1cf6b0f67231141a9226efdb771a27f9281de', 'validated_manifest.csv': '1536c6eb78e2857e599da4f7b3854ed5fa01fd66e50f390dd315d72fe5572aee', 'validation_audit.json': 'f3706a7197e47c1a2693464f9f01452072ac9e6084388966968e37b42d0c3fae', 'decoder_vocabulary.json': 'c0e169c3558fe0cbf2e36431281aa223f64e2e4fb88f7361e23b33e12bcc3ad5', 'mclip_text_features.pt': '7275ebb5ea1a9ef273eee38b16e45fe6a3f42f154bb844ca2ba890e09853863a', 'mclip_prompt_bank.pt': '3cb770a2a470c70d95952828cf23abc088250571897b960a2a39e9bae6c8e1ce'}
RUNTIME_SHA256 = 'e897aa5e800ce18911744cc13e16b75b239477a4edb8bc1df0f8e9ab3f9e038a'
RUNTIME_SOURCE = '"""Checkpoint-producing Kaggle decoder and fail-closed recovery utilities.\n\nThis profile preserves the recorded COV-CTR experiment, not modular defaults.\nNo training, model download, or inference is performed on import.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport re\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nARCHITECTURE = dict(vocab_size=368, max_tokens=160, d_model=768,\n                    num_layers=6, num_heads=8, ffn_dim=3072, dropout=0.1,\n                    activation="gelu", norm_first=False, final_layer_norm=False,\n                    scale_token_embeddings=False, pad_id=0, bos_id=1, eos_id=2)\nTRAINING = dict(seed=42, batch_size=6, learning_rate=0.001, momentum=0.99,\n                gradient_clip=5.0, patience=50, max_epochs=200,\n                precision="float32", scheduler=None,\n                selection_metric="validation_text_reconstruction_BLEU4_PTB_x100")\nMCLIP_NAME = "M-CLIP/XLM-Roberta-Large-Vit-L-14"\nDEPENDENCIES = ("train_manifest.csv", "validation_manifest.csv", "test_manifest.csv",\n                "prompt_manifest.csv", "validated_manifest.csv", "validation_audit.json",\n                "decoder_vocabulary.json", "mclip_text_features.pt", "mclip_prompt_bank.pt")\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef load_checkpoint(path):\n    require(Path(path).is_file(), f"Selected checkpoint does not exist: {path}")\n    # No unsafe unpickling fallback. Legacy project artifacts load with weights_only=True.\n    return torch.load(path, map_location="cpu", weights_only=True)\n\n\nclass CheckpointReportDecoder(nn.Module):\n    def __init__(self, architecture=None):\n        super().__init__()\n        self.architecture = dict(ARCHITECTURE if architecture is None else architecture)\n        a = self.architecture\n        require(not a["scale_token_embeddings"] and not a["final_layer_norm"],\n                "This class implements the recorded Kaggle decoder only")\n        self.token_embedding = nn.Embedding(a["vocab_size"], a["d_model"], padding_idx=a["pad_id"])\n        self.position_embedding = nn.Embedding(a["max_tokens"], a["d_model"])\n        layer = nn.TransformerDecoderLayer(\n            d_model=a["d_model"], nhead=a["num_heads"], dim_feedforward=a["ffn_dim"],\n            dropout=a["dropout"], activation=a["activation"], batch_first=True,\n            norm_first=a["norm_first"])\n        self.transformer = nn.TransformerDecoder(layer, num_layers=a["num_layers"])\n        self.output_projection = nn.Linear(a["d_model"], a["vocab_size"])\n\n    def forward(self, condition, input_ids):\n        a = self.architecture\n        require(input_ids.ndim == 2 and 0 < input_ids.shape[1] <= a["max_tokens"],\n                "Decoder IDs must be [B,T] within the saved positional limit")\n        require(tuple(condition.shape) == (input_ids.shape[0], a["d_model"]),\n                "Condition dimensions incompatible with checkpoint decoder")\n        positions = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)\n        target = self.token_embedding(input_ids) + self.position_embedding(positions)\n        mask = torch.triu(torch.ones(input_ids.shape[1], input_ids.shape[1],\n                                    dtype=torch.bool, device=input_ids.device), diagonal=1)\n        hidden = self.transformer(target, condition.unsqueeze(1).to(target.dtype),\n                                  tgt_mask=mask, tgt_key_padding_mask=input_ids.eq(a["pad_id"]))\n        return self.output_projection(hidden)\n\n    @torch.no_grad()\n    def generate(self, condition, max_length=None):\n        self.eval()\n        a = self.architecture\n        limit = a["max_tokens"] if max_length is None else int(max_length)\n        require(2 <= limit <= a["max_tokens"], "Invalid generation length")\n        ids = torch.full((len(condition), 1), a["bos_id"], dtype=torch.long, device=condition.device)\n        finished = torch.zeros(len(condition), dtype=torch.bool, device=condition.device)\n        for _ in range(limit - 1):\n            next_id = self(condition, ids)[:, -1].argmax(-1)\n            next_id = torch.where(finished, torch.full_like(next_id, a["pad_id"]), next_id)\n            ids = torch.cat((ids, next_id[:, None]), dim=1)\n            finished |= next_id.eq(a["eos_id"])\n            if bool(finished.all()):\n                break\n        return ids\n\n\ndef validate_model_state(checkpoint, model):\n    require(isinstance(checkpoint, dict), "Checkpoint must be a state dictionary payload")\n    state = checkpoint.get("decoder")\n    require(isinstance(state, dict), "Required decoder state dictionary absent")\n    expected = model.state_dict()\n    require(set(state) == set(expected),\n            f"Decoder key mismatch: missing={sorted(set(expected)-set(state))}; unexpected={sorted(set(state)-set(expected))}")\n    for key, tensor in state.items():\n        require(isinstance(tensor, torch.Tensor) and tensor.shape == expected[key].shape,\n                f"Decoder shape mismatch: {key}")\n        require(tensor.dtype == expected[key].dtype, f"Decoder dtype mismatch: {key}")\n        require(bool(torch.isfinite(tensor).all()), f"Non-finite decoder weights: {key}")\n    if "architecture" in checkpoint:\n        require(checkpoint["architecture"] == model.architecture, "Saved architecture metadata differs")\n    for key, expected_value in (("vocabulary_size", model.architecture["vocab_size"]),\n                                ("max_tokens", model.architecture["max_tokens"])):\n        if key in checkpoint:\n            require(checkpoint[key] == expected_value, f"Saved {key} differs")\n\n\ndef inspect_training_state(checkpoint, model, best_checkpoint=None):\n    require(isinstance(checkpoint, dict), "Training checkpoint must be a dictionary payload")\n    required = {"epoch", "decoder", "optimizer", "history", "best_validation_bleu4", "epochs_without_improvement"}\n    require(required <= checkpoint.keys(), f"Required training state absent: {sorted(required-checkpoint.keys())}")\n    epoch = checkpoint["epoch"]\n    require(type(epoch) is int and epoch > 0, "Saved epoch must be a positive integer")\n    require(checkpoint.get("scheduler") is None or checkpoint.get("scheduler") == {},\n            "Unknown scheduler state; supply its verified implementation before resuming")\n    if "training_config" in checkpoint:\n        require(checkpoint["training_config"] == TRAINING, "Training configuration differs from recorded run")\n    history = checkpoint["history"]\n    require(isinstance(history, list) and len(history) == epoch, "History length differs from saved epoch")\n    require(all(isinstance(row, dict) and {"epoch", "training_loss", "validation_bleu4"} <= row.keys()\n                for row in history), "History record fields absent or invalid")\n    require([row["epoch"] for row in history] == list(range(1, epoch + 1)), "History is not contiguous")\n    for row in history:\n        require(all(math.isfinite(float(row[key])) for key in ("training_loss", "validation_bleu4")),\n                "Non-finite history value")\n    selected = max(history, key=lambda row: row["validation_bleu4"])\n    require(math.isclose(float(selected["validation_bleu4"]), float(checkpoint["best_validation_bleu4"]),\n                         rel_tol=0, abs_tol=1e-10), "History and saved best metric disagree")\n    require(type(checkpoint["epochs_without_improvement"]) is int and\n            checkpoint["epochs_without_improvement"] == epoch - selected["epoch"], "Patience state disagrees with history")\n    validate_model_state(checkpoint, model)\n    if best_checkpoint is not None:\n        validate_selected_best(checkpoint, model, best_checkpoint)\n    opt = checkpoint["optimizer"]\n    require(isinstance(opt, dict) and {"state", "param_groups"} <= opt.keys(), "Invalid optimizer state")\n    require(len(opt["param_groups"]) == 1, "Expected one decoder SGD parameter group")\n    group = opt["param_groups"][0]\n    for key, expected in dict(lr=0.001, momentum=0.99, dampening=0, weight_decay=0,\n                              nesterov=False, maximize=False).items():\n        require(group.get(key, False if key == "maximize" else None) == expected,\n                f"SGD setting mismatch: {key}")\n    parameters = list(model.named_parameters())\n    saved_ids = group["params"]\n    require(len(saved_ids) == len(parameters) and len(set(saved_ids)) == len(saved_ids),\n            "Optimizer parameter count or mapping incompatible")\n    require(set(opt["state"]) == set(saved_ids), "Missing or unexpected optimizer parameter states")\n    nonzero = 0\n    for pid, (name, parameter) in zip(saved_ids, parameters):\n        require(set(opt["state"][pid]) == {"momentum_buffer"}, f"Unknown/missing SGD state: {name}")\n        buffer = opt["state"][pid]["momentum_buffer"]\n        require(isinstance(buffer, torch.Tensor) and buffer.shape == parameter.shape and\n                buffer.dtype == parameter.dtype, f"Momentum shape/dtype incompatible: {name}")\n        require(bool(torch.isfinite(buffer).all()), f"Non-finite momentum: {name}")\n        nonzero += int(bool(buffer.ne(0).any()))\n    require(nonzero > 0, "All SGD momentum is zero")\n    if "parameter_names" in checkpoint:\n        require(checkpoint["parameter_names"] == [name for name, _ in parameters], "Optimizer parameter order differs")\n    return dict(epoch=epoch, next_epoch=epoch + 1, best_epoch=selected["epoch"],\n                best_validation_bleu4=float(selected["validation_bleu4"]),\n                epochs_without_improvement=checkpoint["epochs_without_improvement"],\n                momentum_buffers=len(saved_ids), nonzero_momentum_buffers=nonzero,\n                parameter_count=sum(p.numel() for _, p in parameters),\n                legacy_rng_missing="rng_state" not in checkpoint,\n                scheduler="none", compatibility="PASS")\n\n\ndef validate_selected_best(checkpoint, model, best_checkpoint):\n    require(isinstance(best_checkpoint, dict), "Selected best must be a dictionary payload")\n    selected = max(checkpoint["history"], key=lambda row: row["validation_bleu4"])\n    require(best_checkpoint.get("epoch") == selected["epoch"] and\n            math.isclose(float(best_checkpoint.get("validation_bleu4", float("nan"))),\n                         float(selected["validation_bleu4"]), rel_tol=0, abs_tol=1e-10),\n            f"Best checkpoint disagrees with history: need epoch {selected[\'epoch\']}, "\n            f"found epoch {best_checkpoint.get(\'epoch\')}. Recover matching weights; do not substitute latest weights.")\n    validate_model_state(best_checkpoint, model)\n\n\ndef restore_weights_optimizer(checkpoint, model, device="cpu"):\n    """Restore latest state for diagnostics; not authorization to continue training."""\n    result = inspect_training_state(checkpoint, model)\n    model.load_state_dict(checkpoint["decoder"], strict=True)\n    model.to(device)\n    optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.99)\n    optimizer.load_state_dict(checkpoint["optimizer"])\n    for parameter, state in optimizer.state.items():\n        state["momentum_buffer"] = state["momentum_buffer"].to(parameter.device)\n    for pid, parameter in zip(checkpoint["optimizer"]["param_groups"][0]["params"], model.parameters()):\n        require(torch.equal(optimizer.state[parameter]["momentum_buffer"].cpu(),\n                            checkpoint["optimizer"]["state"][pid]["momentum_buffer"].cpu()),\n                "Momentum changed during restoration")\n    model.eval()\n    return optimizer, result\n\n\ndef restore_training(checkpoint, model, best_checkpoint, device="cpu"):\n    result = inspect_training_state(checkpoint, model, best_checkpoint)\n    optimizer, _ = restore_weights_optimizer(checkpoint, model, device)\n    return optimizer, result\n\n\ndef read_manifest(directory, name):\n    with (Path(directory) / name).open(encoding="utf-8-sig", newline="") as handle:\n        return list(csv.DictReader(handle))\n\n\ndef validate_artifacts(directory, expected_hashes=None):\n    directory = Path(directory)\n    hashes = {name: sha256(directory / name) for name in DEPENDENCIES}\n    if expected_hashes is not None:\n        require(hashes == expected_hashes, "Feature/vocabulary/split dependency hashes differ")\n    records = {name: read_manifest(directory, name + "_manifest.csv")\n               for name in ("train", "validation", "test", "prompt", "validated")}\n    ids = {}\n    for name, count in dict(train=571, validation=71, test=72, prompt=500, validated=714).items():\n        rows = records[name]\n        ids[name] = [row["image_id"] for row in rows]\n        require(len(rows) == count and len(set(ids[name])) == count, f"{name} count/ID uniqueness failed")\n    train, val, test = (set(ids[name]) for name in ("train", "validation", "test"))\n    require(not (train & val or train & test or val & test), "Split leakage")\n    require(train | val | test == set(ids["validated"]), "Split population differs")\n    require(set(ids["prompt"]) <= train, "Prompt bank contains non-training IDs")\n    by_id = {r["image_id"]: r for r in records["validated"]}\n    for name in ("train", "validation", "test", "prompt"):\n        require(all(row["report"] == by_id[row["image_id"]]["report"] for row in records[name]),\n                f"{name} reports differ from validated manifest")\n    audit = json.loads((directory / "validation_audit.json").read_text(encoding="utf-8"))\n    require(audit["csv_rows"] == 726 and audit["physical_images"] == 746, "Raw audit population differs")\n    vocabulary = json.loads((directory / "decoder_vocabulary.json").read_text(encoding="utf-8"))\n    require(vocabulary["max_tokens"] == 160 and len(vocabulary["vocabulary"]) == 368,\n            "Vocabulary or positional capacity differs")\n    require(vocabulary["special_token_ids"] == dict(pad=0, bos=1, eos=2, unk=3), "Special-token IDs differ")\n    expected_vocab = ["<pad>", "<bos>", "<eos>", "<unk>"] + sorted({\n        token for row in records["train"] for token in re.findall(r"\\w+|[^\\w\\s]", row["report"])})\n    require(vocabulary["vocabulary"] == expected_vocab, "Vocabulary is not the saved training-only lexicographic mapping")\n    text = load_checkpoint(directory / "mclip_text_features.pt")\n    require(text["model"] == MCLIP_NAME and text["normalization"] == "l2", "Text encoder metadata differs")\n    for name in ("train", "validation"):\n        require(set(text[name]) == set(ids[name]), f"{name} feature membership differs")\n        for vector in text[name].values():\n            require(vector.shape == (768,) and bool(torch.isfinite(vector).all()) and\n                    torch.allclose(vector.norm(), torch.tensor(1.0), atol=1e-4), "Invalid normalized text feature")\n    prompt = load_checkpoint(directory / "mclip_prompt_bank.pt")\n    require(prompt["model"] == MCLIP_NAME and prompt.get("normalization") == "raw", "Prompt encoder/normalization differs")\n    require(prompt["prompt_ids"] == ids["prompt"], "Prompt tensor row order/provenance differs")\n    vectors = prompt["embeddings"]\n    require(vectors.shape == (500, 768) and bool(torch.isfinite(vectors).all()) and\n            bool((vectors.norm(dim=-1) > 0).all()), "Invalid prompt bank")\n    aligned = torch.stack([text["train"][key] for key in ids["prompt"]])\n    # Separate encodings are not bitwise equal. Record the observed difference;\n    # membership and frozen-cache hashes above are the provenance assertions.\n    normalized_prompts = F.normalize(vectors, dim=-1)\n    cosine = (normalized_prompts * aligned).sum(-1)\n    cache_comparison = dict(max_absolute_difference=float((normalized_prompts - aligned).abs().max()),\n                            minimum_cosine=float(cosine.min()), mean_cosine=float(cosine.mean()),\n                            exact_encoder_revision="UNVERIFIED")\n    return dict(records=records, vocabulary=vocabulary, text=text, prompt=prompt,\n                dependency_hashes=hashes, prompt_text_comparison=cache_comparison)\n\n\ndef encode_report(text, vocabulary):\n    lookup = {token: i for i, token in enumerate(vocabulary)}\n    ids = [1] + [lookup.get(token, 3) for token in re.findall(r"\\w+|[^\\w\\s]", text)] + [2]\n    require(len(ids) <= 160, "Report exceeds saved positional capacity; no silent truncation")\n    return ids\n\n\ndef decode_report(token_ids, vocabulary):\n    tokens = []\n    for raw in token_ids:\n        token = int(raw)\n        if token == 2:\n            break\n        require(0 <= token < len(vocabulary), "Generated token outside vocabulary")\n        if token not in {0, 1}:\n            tokens.append(vocabulary[token])\n    text = re.sub(r"\\s+([.,!?;:%)\\]])", r"\\1", " ".join(tokens))\n    return re.sub(r"([(\\[])\\s+", r"\\1", text).strip()\n\n\ndef enrich(image, prompts):\n    require(image.ndim == prompts.ndim == 2 and image.shape[-1] == prompts.shape[-1] == 768,\n            "Image/prompt dimensions must share the recorded 768-D CLIP space")\n    require(bool(torch.isfinite(image).all()) and bool(torch.isfinite(prompts).all()), "Non-finite encoder output")\n    scores = image @ prompts.T / math.sqrt(768)\n    attention = scores.softmax(-1)\n    residual = image + attention @ prompts\n    require(bool((residual.norm(dim=-1) > 0).all()), "Zero residual embedding")\n    phi = F.normalize(residual, p=2, dim=-1)\n    require(bool(torch.isfinite(phi).all()), "Non-finite enriched features")\n    return phi, attention\n\n\ndef capture_rng():\n    return dict(python=random.getstate(), torch_cpu=torch.get_rng_state(),\n                torch_cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])\n\n\ndef restore_rng(state):\n    random.setstate(state["python"])\n    torch.set_rng_state(state["torch_cpu"].cpu())\n    if state["torch_cuda"]:\n        require(len(state["torch_cuda"]) == torch.cuda.device_count(), "CUDA RNG device count differs")\n        torch.cuda.set_rng_state_all([value.cpu() for value in state["torch_cuda"]])\n\n\ndef atomic_save(payload, destination):\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_suffix(destination.suffix + ".tmp")\n    with temporary.open("wb") as handle:\n        torch.save(payload, handle)\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(temporary, destination)\n\n\ndef training_payload(model, optimizer, epoch, history, best_metric, bad_epochs, hashes, provenance):\n    return dict(schema_version="zeromrg_covctr_gate1_v1", epoch=epoch, decoder=model.state_dict(),\n                optimizer=optimizer.state_dict(), scheduler=None, history=history,\n                best_validation_bleu4=best_metric, epochs_without_improvement=bad_epochs,\n                architecture=model.architecture, parameter_names=[n for n, _ in model.named_parameters()],\n                training_config=TRAINING, dependency_hashes=hashes, provenance=provenance,\n                rng_state=capture_rng())\n'

In [ ]:
import gc, platform, shutil, sys, tempfile, time, unicodedata, uuid, zipfile
from collections import Counter, defaultdict
from importlib import metadata

# Select an exact checkpoint; no automatic newest-file selection or epoch assumption.
CHECKPOINT_PATH = Path('/kaggle/input/datasets/inshasadaf114/zeromrg-checkpoint-backup5-zip/text_decoder_last.pt')
ARTIFACT_DIR = CHECKPOINT_PATH.parent
BEST_CHECKPOINT_PATH = ARTIFACT_DIR / 'text_decoder_best.pt'
DATA_ROOT = Path('/kaggle/input/datasets/inshasadaf114/dataset/COV-CTR with English reports')
RUN_ROOT = Path('/kaggle/working') / ('zeromrg_gate1_' + uuid.uuid4().hex[:12])
require(Path('/kaggle/input').is_dir() and Path('/kaggle/working').is_dir(), 'Execute this notebook on Kaggle only')
require(torch.cuda.is_available(), 'Enable a Kaggle GPU; no local CPU fallback')
DEVICE = torch.device('cuda:0')
require(CHECKPOINT_PATH.is_file(), 'Set CHECKPOINT_PATH to the exact mounted latest checkpoint')
require(DATA_ROOT.is_dir(), 'Set DATA_ROOT to the attached COV-CTR directory')
require(RUN_ROOT.resolve().is_relative_to(Path('/kaggle/working').resolve()), 'Output must stay in /kaggle/working')
RUN_ROOT.mkdir(exist_ok=False)
write_probe = RUN_ROOT / 'write_probe.txt'
write_probe.write_text('writable', encoding='utf-8')
require(write_probe.read_text(encoding='utf-8') == 'writable', 'Output write/read failed')
print('Mounted roots:', [str(p) for p in Path('/kaggle/input').iterdir()])
gpu = torch.cuda.get_device_properties(0)
free_memory, total_memory = torch.cuda.mem_get_info()
versions = {'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda}
for distribution in ('numpy', 'pandas', 'Pillow', 'clip', 'multilingual-clip', 'transformers', 'pycocoevalcap'):
    try:
        versions[distribution] = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        versions[distribution] = 'not installed'
print('GPU:', gpu.name, '| free/total GiB:', free_memory/2**30, total_memory/2**30)
print('Versions:', versions)
print('Selected checkpoint:', CHECKPOINT_PATH, '| output:', RUN_ROOT)

In [ ]:
# Legacy recovery must match the audited dependencies exactly. New checkpoints carry
# the same dependency hashes; changed splits/features are refused.
checkpoint = load_checkpoint(CHECKPOINT_PATH)
expected_dependencies = checkpoint.get('dependency_hashes', CANONICAL_DEPENDENCY_HASHES)
artifacts = validate_artifacts(ARTIFACT_DIR, expected_dependencies)
records = artifacts['records']
vocabulary = artifacts['vocabulary']['vocabulary']
text_cache = artifacts['text']
prompt_cache = artifacts['prompt']
external_history = json.loads((ARTIFACT_DIR / 'text_training_history.json').read_text(encoding='utf-8'))
require(external_history == checkpoint['history'], 'External and checkpoint history disagree')

# Verify mounted raw population and saved target reports without rebuilding splits.
csv_candidates = list(DATA_ROOT.rglob('reports_ZH_EN.csv'))
require(len(csv_candidates) == 1, 'Expected one attached COV-CTR CSV')
with csv_candidates[0].open(encoding='utf-8-sig', newline='') as handle:
    raw_rows = list(csv.DictReader(handle))
require(len(raw_rows) == 726, 'Expected 726 raw report rows')
images = [p for p in DATA_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in
          {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}]
require(len(images) == 746, 'Expected 746 physical image files')
raw_ids = Counter(row['image_id'].strip() for row in raw_rows)
require(sum(count > 1 for count in raw_ids.values()) == 6, 'Duplicate-ID audit differs')
raw_by_id = {row['image_id'].strip(): row for row in raw_rows if raw_ids[row['image_id'].strip()] == 1}
image_index = defaultdict(list)
for p in images:
    image_index[p.name.lower()].append(p)
for row in records['validated']:
    key = row['image_id']
    normalized = re.sub(r'\s+', ' ', unicodedata.normalize('NFC', raw_by_id[key]['reports_En'])).strip()
    require(normalized == row['report'], f'Saved report differs from raw source: {key}')
    require(len(image_index[key.lower()]) == 1, f'Image missing/ambiguous: {key}')
print('Dataset/split counts: 726 rows, 746 images, 714 usable; 571/71/72 partitions')
print('Training-only vocabulary: 368; prompt bank: 500 x 768 raw; normalized text: 571 + 71')

def token_batch(rows):
    sequences = [encode_report(row['report'], vocabulary) for row in rows]
    ids = torch.full((len(rows), max(map(len, sequences))), 0, dtype=torch.long, device=DEVICE)
    for index, sequence in enumerate(sequences):
        ids[index, :len(sequence)] = torch.tensor(sequence, device=DEVICE)
    return ids

def feature_batch(rows, name):
    return torch.stack([text_cache[name][row['image_id']] for row in rows]).to(DEVICE)

def decoder_loss(condition, ids):
    logits = decoder(condition, ids[:, :-1])
    return F.cross_entropy(logits.reshape(-1, 368), ids[:, 1:].reshape(-1),
                           ignore_index=0, reduction='sum') / ids[:, 1:].ne(0).sum()

In [ ]:
with torch.device('meta'):
    decoder = CheckpointReportDecoder()
latest_inspection = inspect_training_state(checkpoint, decoder)
decoder.load_state_dict(checkpoint['decoder'], strict=True, assign=True)
optimizer, latest_inspection = restore_weights_optimizer(checkpoint, decoder, DEVICE)
print('Latest model/optimizer:', latest_inspection)
print('Saved best epoch from history:', latest_inspection['best_epoch'], '(weights still require verification)')

# One TRAINING sample; no optimizer.step, backward, validation scoring, or test inference.
probe_rows = records['train'][:1]
decoder.eval()
with torch.no_grad():
    conditions = feature_batch(probe_rows, 'train')
    ids = token_batch(probe_rows)
    probe_logits = decoder(conditions, ids[:, :-1])
    probe_loss = decoder_loss(conditions, ids)
require(bool(torch.isfinite(probe_logits).all()) and bool(torch.isfinite(probe_loss)), 'Non-finite diagnostic output')
require(all(parameter.grad is None for parameter in decoder.parameters()), 'Unexpected diagnostic gradients')
print('Finite latest-checkpoint TRAIN forward CE:', float(probe_loss))

# Disposable full model + optimizer round trip, in an isolated temporary directory.
with tempfile.TemporaryDirectory(prefix='disposable_restore_', dir=RUN_ROOT) as temporary:
    diagnostic_path = Path(temporary) / 'diagnostic.pt'
    atomic_save(checkpoint, diagnostic_path)
    reloaded = load_checkpoint(diagnostic_path)
    inspect_training_state(reloaded, decoder)
    for key in checkpoint['decoder']:
        require(torch.equal(reloaded['decoder'][key], checkpoint['decoder'][key]), 'Diagnostic weights changed')
    for pid in checkpoint['optimizer']['state']:
        require(torch.equal(reloaded['optimizer']['state'][pid]['momentum_buffer'],
                            checkpoint['optimizer']['state'][pid]['momentum_buffer']), 'Diagnostic momentum changed')
    optimizer, roundtrip_inspection = restore_weights_optimizer(reloaded, decoder, DEVICE)
    with torch.no_grad():
        restored_logits = decoder(conditions, ids[:, :-1])
    require(torch.equal(restored_logits, probe_logits), 'Forward output changed after diagnostic restoration')
    saved_hash = sha256(diagnostic_path)
    del reloaded
print('Disposable save/restore PASS; SHA-256:', saved_hash)
gc.collect()

## Validate the selected best and frozen image path

The latest checkpoint diagnostic above is separate from validation-selected zero-shot readiness. With current inputs the next cell writes a BLOCKED report and stops before loading CLIP. After the correct best artifact is recovered, it uses one fixed training image, with no metrics or tuning.

In [ ]:
# Gate 0's epoch-85 history entry is NOT evidence of epoch-85 model weights.
# The audited backup actually contains an epoch-8 best artifact. Report this clearly.
preflight = dict(kind='preflight_only', checkpoint_sha256=sha256(CHECKPOINT_PATH),
                 latest=latest_inspection, dependency_assertions='PASS',
                 finite_train_forward_loss=float(probe_loss), disposable_restore='PASS',
                 versions=versions, gpu=gpu.name, zero_shot='BLOCKED', overall='BLOCKED')
best = load_checkpoint(BEST_CHECKPOINT_PATH)
try:
    validate_selected_best(checkpoint, decoder, best)
    preflight['best_selection'] = 'PASS'
except ValueError as error:
    preflight['best_selection'] = str(error)
    (RUN_ROOT / 'preflight_report.json').write_text(json.dumps(preflight, indent=2), encoding='utf-8')
    print(json.dumps(preflight, indent=2))
    raise ValueError('PREFLIGHT BLOCKED: validation-selected best weights unavailable. Latest restoration checks passed.') from error

# Reached only with a verified best artifact. Never substitute epoch-90 weights here.
# If absent, install ONLY the official OpenAI CLIP package on Kaggle, then rerun:
# %pip install 'clip @ git+https://github.com/openai/CLIP.git'
import clip
from PIL import Image
decoder.load_state_dict(best['decoder'], strict=True)
decoder.eval()
for parameter in decoder.parameters():
    parameter.requires_grad_(False)
clip_model, clip_preprocess = clip.load('ViT-L/14', device=DEVICE, jit=False)
vision = clip_model.visual.eval()
for parameter in vision.parameters():
    parameter.requires_grad_(False)
del clip_model
require(all(not p.requires_grad for p in vision.parameters()), 'Image encoder must be frozen')
require(all(not p.requires_grad for p in decoder.parameters()), 'Decoder must be frozen for zero-shot')
require(int(vision.output_dim) == 768, 'Image embedding dimension differs')
probe_image_path = image_index[probe_rows[0]['image_id'].lower()][0]
with Image.open(probe_image_path) as image:
    image_tensor = clip_preprocess(image.convert('RGB')).unsqueeze(0)
require(tuple(image_tensor.shape) == (1, 3, 224, 224), 'CLIP image preprocessing shape differs')
prompts = prompt_cache['embeddings'].to(DEVICE, dtype=torch.float32)
require(not prompts.requires_grad, 'Prompt bank must be frozen')
with torch.no_grad():
    image_embedding = vision(image_tensor.to(DEVICE, dtype=vision.conv1.weight.dtype)).float()
    phi, attention = enrich(image_embedding, prompts)
    require(torch.allclose(attention.sum(-1), torch.ones(1, device=DEVICE), atol=1e-5), 'Attention normalization failed')
    require(torch.allclose(phi.norm(dim=-1), torch.ones(1, device=DEVICE), atol=1e-5), 'Enriched feature normalization failed')
    image_logits = decoder(phi, torch.tensor([[1]], device=DEVICE))
    require(bool(torch.isfinite(image_logits).all()), 'Image-conditioned logits non-finite')
    generated = decoder.generate(phi, max_length=160).cpu()
preflight.update(zero_shot='PASS: one training image, no scoring', overall='PASS',
                 best_checkpoint_sha256=sha256(BEST_CHECKPOINT_PATH),
                 clip_weight_url=clip.clip._MODELS['ViT-L/14'],
                 diagnostic_image_id=probe_rows[0]['image_id'],
                 diagnostic_generation=decode_report(generated[0], vocabulary),
                 generated_length=int(generated.shape[1]))
(RUN_ROOT / 'preflight_report.json').write_text(json.dumps(preflight, indent=2), encoding='utf-8')
print(json.dumps(preflight, indent=2))
print('PREFLIGHT PASS; no training or held-out test evaluation was performed')